# **NYU ECE-GY 9483 Spring 2026: Lab 2 Quantization**



Hey everyone, this lab focuses on neural network quantization. The lab contains five parts, with a total of 100 points:
1. Implement fixed-point quantization function: tensor-wise and filter-wise asymmetric post-training quantization (30 pts)
2. Implement floating point quantization (10 pts)
3. Hyperparameter Selection: adjust the clipping range for the 4-bit weight quantization and investigate its impact on accuracy (20 points)  
4. Understand the Straight Through Estimator (STE) approximation and implement quantization-aware training (15 pts)
5. Explore transformer-based language models and implement tensor-wise, channel-wise, and token-wise quantization functions (25 pts)

If you're using Colab, go to the settings menu (Runtime -> Change runtime type) and select GPU as the hardware accelerator. The best way to run your code is using Colab and it's free.

Same as Lab1, you need to solve two types of problems: coding and short-answer questions.

* **Coding: you only need to complete the section labeled as:**   
```
    ##############################################################################
    #TODO:
    ##############################################################################

    ##############################################################################
    #END OF YOUR CODE                              #
    ##############################################################################
```
* **Question Answering: questions start with a red dot "🔴", and you should write your answers in the following text block.**


***Please upload your ipynb file for Lab 2 to Brightspace. Name the file as "netid_lab_2.ipynb" and ensure it is submitted by Mar 15th at 11:59 PM.***

# GPU Requirement

This assignment requires **CUDA support**. Please make sure to run it on a **local machine with a GPU** or on **Google Colab** with GPU enabled. We recommend using **Google Colab**, as it provides free access to GPUs.

If you are working on Colab, click the 'Connect' button in the top-right corner. Once connected, go to Runtime->Change runtime type and select GPU as the hardware accelerator.

Once you are connected to a GPU (either on local machine or Colab), you can run the following command to check the GPU on your machine.

In [ ]:
! nvidia-smi

# Initialization

In Lab 2, we use the same ResNet CNN as in Lab 1. However, this time, we apply quantization to both the weights and input activations to analyze its impact on model performance.

Let's start with importing the necessary packages:

In [ ]:
! pip install thop

In [ ]:
import torch
import time
import math
import copy
import random
import logging
import requests
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from tqdm import tqdm
from torch.utils.data import DataLoader
from thop import profile
from thop import clever_format
from torchsummary import summary

assert torch.cuda.is_available()
logging.getLogger('thop').setLevel(logging.WARNING)

In [ ]:
random.seed(10)
np.random.seed(10)
torch.manual_seed(10)

Next, we define the ResNet model:

In [ ]:
class BasicBlock(nn.Module):
    """
    BasicBlock is the fundamental building block of ResNet.
    It consists of two 3x3 convolutional layers.
    If the input dimensions do not match the output dimensions,
    or the stride is not 1, a shortcut (identity mapping) is used to adjust dimensions.
    """

    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        # First convolution layer with batch normalization
        self.conv1 = nn.Conv2d(
            in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(planes)

        # Second convolution layer with batch normalization
        self.conv2 = nn.Conv2d(
            planes, planes, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(planes)

        # Shortcut connection (identity mapping)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_planes,
                    self.expansion * planes,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(self.expansion * planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        shortcut_out = self.shortcut(x)
        out += shortcut_out
        out = F.relu(out)
        return out


class Bottleneck(nn.Module):
    """
    Bottleneck is the advanced building block used in deeper versions of ResNet (e.g., ResNet-50, ResNet-101).
    It consists of three layers: 1x1 conv, 3x3 conv, and another 1x1 conv.
    The purpose of the bottleneck is to reduce computation by compressing the dimensions before the 3x3 convolution.
    """

    expansion = 4

    def __init__(self, in_planes, planes, stride=1):
        super(Bottleneck, self).__init__()
        # 1x1 convolution to reduce the number of input channels
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)

        # 3x3 convolution for main processing
        self.conv2 = nn.Conv2d(
            planes, planes, kernel_size=3, stride=stride, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(planes)

        # 1x1 convolution to expand the number of output channels
        self.conv3 = nn.Conv2d(
            planes, self.expansion * planes, kernel_size=1, bias=False
        )
        self.bn3 = nn.BatchNorm2d(self.expansion * planes)

        # Shortcut connection
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_planes,
                    self.expansion * planes,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(self.expansion * planes),
            )

    def forward(self, x):
        # Forward pass through the bottleneck block (1x1 -> 3x3 -> 1x1)
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))

        # Add the shortcut connection and pass through ReLU again
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNet(nn.Module):
    """
    ResNet class that builds the full ResNet architecture using the BasicBlock or Bottleneck block.
    The number of blocks and layers is determined by the `num_blocks` parameter.
    """

    def __init__(self, block, num_blocks, num_classes=10):
        super(ResNet, self).__init__()
        self.in_planes = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        # Create the subsequent layers of the network using the blocks
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)

        # Final fully connected layer to classify the output
        self.linear = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        """
        Creates a residual layer consisting of `num_blocks` residual blocks.
        The first block may have a stride > 1 to downsample the feature maps.
        """
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))

        # Pass through the 4 layers of the network
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)

        # Flatten the output and pass through the fully connected layer
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out


def ResNet18():
    return ResNet(BasicBlock, [2, 2, 2, 2])


def ResNet34():
    return ResNet(BasicBlock, [3, 4, 6, 3])


def ResNet50():
    return ResNet(Bottleneck, [3, 4, 6, 3])


def ResNet101():
    return ResNet(Bottleneck, [3, 4, 23, 3])


def ResNet152():
    return ResNet(Bottleneck, [3, 8, 36, 3])


In [ ]:
def train(model: nn.Module, train_dataloader: DataLoader, lr=0.1, epochs=10, device="cuda"):
    """
    Training function for training the model using the given dataloader, optimizer, and loss function.

    Args:
        model: The neural network model to be trained.
        train_dataloader: The DataLoader for the training data.
        lr: The learning rate for the optimizer.
        epochs: Number of epochs to train the model.
        device: Device to run the training on ('cuda' or 'cpu').

    Returns:
        None. Prints training progress for each epoch.
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(
        model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs
    )

    for epoch in range(epochs):
        print(f"\nEpoch: {epoch+1}/{epochs}")
        model.train()
        train_loss = 0
        correct = 0
        total = 0

        with tqdm(total=len(train_dataloader), desc=f"Train Epoch {epoch+1}") as pbar:
            for batch_idx, (inputs, targets) in enumerate(train_dataloader):
                inputs, targets = inputs.to(device), targets.to(device)

                optimizer.zero_grad()
                outputs = model(inputs)

                # Compute the loss
                loss = criterion(outputs, targets)

                # Backward pass (backpropagation)
                loss.backward()

                # Optimizer step (update model parameters)
                optimizer.step()

                # Update the running loss and accuracy
                train_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()

                pbar.set_postfix(
                    {
                        "Loss": f"{train_loss/(batch_idx+1):.3f}",
                        "Acc": f"{100.0 * correct / total:.3f}%",
                    }
                )
                pbar.update(1)
        scheduler.step()

        epoch_loss = train_loss / len(train_dataloader)
        epoch_acc = 100.0 * correct / total
        print(
            f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%"
        )


In [ ]:
def evaluate(model: nn.Module, test_dataloader: DataLoader, device="cuda") -> float:
    """
    Inference function to evaluate the model on a test dataset using the provided dataloader,
    with tqdm progress bar for visualization.

    Args:
        model: The neural network model to be used for inference.
        test_dataloader: The dataloader for the test data.
        device: Device to use ('cuda' or 'cpu').

    Returns:
        None: Prints the accuracy on the test set.
    """
    model = model.to(device)
    model.eval()

    correct = 0
    total = 0

    # Disable gradient computation during inference
    with torch.no_grad():
        for inputs, targets in tqdm(test_dataloader, desc="Evaluating", leave=False):
            inputs, targets = inputs.to(device), targets.to(device)

            outputs = model(inputs)
            _, predicted = outputs.max(1)

            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    accuracy = 100.0 * correct / total
    # print(f"Accuracy on the test set: {accuracy:.2f}%")
    return accuracy

Download and model initialization function

In [ ]:
def download_and_load_model(url: str, weights_path: str, device="cuda"):
    """
    Downloads the model weights from the given URL and loads them into a ResNet18 model.

    Args:
        url (str): URL to download the model weights from.
        weights_path (str): Path to save the downloaded weights.
        device (str): Device to load the model onto ('cuda' or 'cpu').

    Returns:
        model (torch.nn.Module): The ResNet18 model with the loaded weights.
        recover_model (lambda): A lambda function to recover the model's state.
    """
    print(f"Downloading weights from {url}...")
    r = requests.get(url, allow_redirects=True)
    open(weights_path, 'wb').write(r.content)

    model = ResNet18()

    print(f"Loading weights from {weights_path}...")
    checkpoint = torch.load(weights_path, map_location=torch.device(device))

    if 'state_dict' in checkpoint:
        model.load_state_dict(checkpoint['state_dict'])
    else:
        model.load_state_dict(checkpoint)

    def recover_model():
        print("Recovering model to its original state.")
        if 'state_dict' in checkpoint:
            model.load_state_dict(checkpoint['state_dict'], strict = False)
        else:
            model.load_state_dict(checkpoint, strict=False)

    model = model.to(device)
    return model, recover_model

In [ ]:
url = "https://github.com/shawnyin128/NYU-EfficientAI-Materials/raw/master/model/best_model.pth?raw=true"
weights_path = "./best_model.pth"

model, recover_model = download_and_load_model(url, weights_path, device="cuda")

Now we load the CIFAR-10 dataset

In [ ]:
class CIFAR:
    """
    Initialize CIFAR dataset (CIFAR-10 in this case).
    This class prepares the train and test data loaders.
    """

    def __init__(self, batch_size=128, num_workers=2):
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.train_loader, self.test_loader = self.prepare_data()

    def prepare_data(self):
        """
        Prepare train and test data loaders with transforms.
        """
        transform_train = transforms.Compose(
            [
                transforms.RandomCrop(32, padding=4),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(
                    (0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)
                ),
            ]
        )

        transform_test = transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Normalize(
                    (0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)
                ),
            ]
        )

        trainset = torchvision.datasets.CIFAR10(
            root="./data", train=True, download=True, transform=transform_train
        )
        trainloader = torch.utils.data.DataLoader(
            trainset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
        )

        testset = torchvision.datasets.CIFAR10(
            root="./data", train=False, download=True, transform=transform_test
        )
        testloader = torch.utils.data.DataLoader(
            testset, batch_size=128, shuffle=False, num_workers=self.num_workers
        )

        return trainloader, testloader

In [ ]:
dataloader = {}
dataloader['train'], dataloader['test'] = CIFAR().train_loader, CIFAR().test_loader

Some useful helper function to calculate statistics.

In [ ]:
accuracy = evaluate(model, dataloader['test'])
print(f'Accuracy: {accuracy}%, ')

From the evaluation results, our model achieve a 93.7% accuracy in the CIFAR-10 test dataset. However, for a relatively simple task CIFAR-10 classification, this model is arguably larger than necessary. Reducing the model's size without sacrificing much accuracy could improve its efficiency, making it more suitable for deployment on resource-constrained devices like mobile phones or embedded systems.

# Question 1: Post-Training Quantization (PTQ) with Fixed-point Format (30 pts)



Given a floating-point tensor $x$ with a clipping range of $x_{\min}$ and $x_{\max}$, asymmetric fixed-point quantization maps $x$ to an integer range $[q_{\min}, q_{\max}]$. For a bitwidth of b, $[q_{\min}, q_{\max}] = [0, 2^b-1]$

The scale factor $s$ is computed as:

$$
s = \frac{x_{\max} - x_{\min}}{q_{\max} - q_{\min}}.
$$

The clipped value \( $x_c$ \) is obtained by:
$$
x_c = \operatorname{clip}\left({x},\; x_{\min},\; x_{\max}\right).
$$

Then, the int value \( $x_{int}$ \) is obtained by:
$$
x_{int} = \operatorname{round}\left(\frac{x_c - x_{\min}}{s}\right).
$$

Finally, the dequantized value \( $x_{q}$ \) is:

$$
x_{q} = x_{int} \cdot s + x_{\min}.
$$




In this section, you will learn how to implement INT quantization and floating point quantization on different bitwidths for both activation and weight quantization:

1. Implement INT quantization function: tensor-wise asymmetric quantization (10 pts)
2. Implement INT quantization function: filter-wise asymmetric quantization (10 pts)
3. Discuss the impact on the accuracy between tensor-wise and filterwise quantization (10 pts)  

## 1.1: Tensor-wise Asymmetric Quantization (10 pts)

You need to implement tensor-wise asymmetric quantization for CNN filters with dimensions $C_{out} \times C_{in} \times W \times H$, where $C_{out}, C_{in}, W$, and $H$ represent the number of filters, input channels, kernel width, and kernel height, respectively. Tensor-wise quantization treats all the layer weights as a single unit and applies quantization on it.

*For simplicity, in 1.1, you can assume the quantization clipping range matches the maximum and minimum values exactly, no clipping is necessary.*

In [ ]:
###############################################
# Quantization Function Implementations
###############################################

import torch

def tensor_wise_quantize(x: torch.Tensor, num_bits: int) -> torch.Tensor:
    ##############################################################################
    #TODO: Implement tensor-wise quantization for CNN
    ##############################################################################
    # step 1: For simplicity, you can assume the quantization clipping range matches the maximum and minimum values exactly, no clipping is necessary.
    x_min = x.min()
    x_max = x.max()
    # step 2: Calculate the scale s
    q_min = 0
    q_max = 2 ** num_bits - 1
    s = (x_max - x_min) / (q_max - q_min)
    # step 3: Calculate the fixed-point representation x_{int}
    x_int = torch.round((x - x_min) / s)
    x_int = torch.clamp(x_int, q_min, q_max)
    # step 4: Rescale
    x_quant = x_int * s + x_min
    ##############################################################################
    #END OF YOUR CODE                              #
    ##############################################################################
    return x_quant

## 1.2: Filter-wise Asymmetric Quantization (10 pts)

<p align="center">
  <img src="https://github.com/shawnyin128/25Fall-CSCI-GA-3033-EfficientAI-Materials/raw/master/graph/lab2_cnn.png" alt="cnn" width="400"/>
</p>

In this section, you need to implement filter-wise asymmetric quantization. Unlike tensor-wise quantization, filter-wise quantization assigns unique quantization parameters (e.g., scale, clipping range) to each filter individually, so each filter will have a separate quantization hyperparameters.

In [ ]:
def filter_wise_quantize(x: torch.Tensor, num_bits: int, channel_axis: int = 0) -> torch.Tensor:
    ##############################################################################
    #TODO: Implement filter-wise quatization for CNN
    ##############################################################################
    # step 1: Assume the quantization clipping range matches the maximum and minimum values exactly, no clipping is necessary.
    num_channels = x.shape[channel_axis]
    # Create shape for broadcasting: e.g., for channel_axis=0 and 4D tensor -> (N, 1, 1, 1)
    shape = [1] * x.dim()
    shape[channel_axis] = num_channels
    # Flatten all dims except channel_axis to compute per-channel min/max
    x_flat = x.flatten(start_dim=channel_axis + 1) if channel_axis < x.dim() - 1 else x.unsqueeze(-1)
    if channel_axis > 0:
        x_flat = x_flat.flatten(end_dim=channel_axis - 1) if channel_axis > 0 else x_flat
    # Reshape to (num_channels, -1)
    x_reshaped = x.view(num_channels, -1) if channel_axis == 0 else x.transpose(0, channel_axis).contiguous().view(num_channels, -1)
    x_min = x_reshaped.min(dim=1)[0].view(shape)
    x_max = x_reshaped.max(dim=1)[0].view(shape)
    # step 2: Calculate the scale s, the scale will be different for each weight filter
    q_min = 0
    q_max = 2 ** num_bits - 1
    s = (x_max - x_min) / (q_max - q_min)
    # step 3: Calculate the fixed-point representation x_{int}
    x_int = torch.round((x - x_min) / s)
    x_int = torch.clamp(x_int, q_min, q_max)
    # step 4: Rescale
    x_quant = x_int * s + x_min
    ##############################################################################
    #END OF YOUR CODE                              #
    ##############################################################################
    return x_quant

## 1.3: Evaluation (10 pts)

Now that you have finished the quantization functions, let's evaluate the accuracy of quantized ResNet18 under different granularity and bit width.

In [ ]:
###############################################
# Post-Training Quantization: Quantize the Model and Evaluate Different Bit-width Combinations
###############################################

def quantize_model(model: nn.Module, quantize_fn, w_bit: int, a_bit: int, mode: str = 'tensor') -> nn.Module:
    """
    Post-Training Quantization of the model:
    - For Conv2d and Linear layers, first quantize the weights.
    - Register a forward hook to quantize the activation outputs.

    Parameter mode: If 'filter', then the weights use channel-wise quantization; otherwise, use quantize_fn (e.g., tensor-wise or floating point).
    """
    model_quant = copy.deepcopy(model)
    for name, module in model_quant.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            # Quantize weights
            if mode == 'filter':
                module.weight.data = filter_wise_quantize(module.weight.data, w_bit, channel_axis=0)
            else:
                module.weight.data = quantize_fn(module.weight.data, w_bit)

            # Register a forward hook to quantize the activation inputs
            def forward_pre_hook(module, input, a_bit=a_bit, quantize_fn=quantize_fn):
                new_input = tuple(quantize_fn(x, a_bit) if isinstance(x, torch.Tensor) else x for x in input)
                return new_input

            module.register_forward_pre_hook(forward_pre_hook)
    return model_quant


def evaluate_quantized_models(model: nn.Module, dataloader: dict, quantize_fn, mode: str, bit_choices = [2, 4, 8]) -> dict:
    """
    Iterate over different bit-width combinations for activations and weights, evaluate the quantized model,
    and return the results as a dictionary.
    """
    results = {}
    for a_bit in bit_choices:
        for w_bit in bit_choices:
            model_q = quantize_model(model, quantize_fn, w_bit, a_bit, mode=mode)
            acc = evaluate(model_q, dataloader['test'])
            results[(a_bit, w_bit)] = acc
            print(f"Quantization mode: {mode} wise, Activation {a_bit} bit, Weight {w_bit} bit -> Accuracy: {acc:.2f}%")
    return results


def plot_quantization_results(results: dict, title: str):
    """
    Visualize the test accuracy of the 9 bit-width combinations as a heatmap.
    The x-axis represents the weight bit-width, and the y-axis represents the activation bit-width.
    """
    import matplotlib.pyplot as plt
    bit_choices = sorted({k[0] for k in results.keys()})
    acc_matrix = np.zeros((len(bit_choices), len(bit_choices)))
    for i, a_bit in enumerate(bit_choices):
        for j, w_bit in enumerate(bit_choices):
            acc_matrix[i, j] = results[(a_bit, w_bit)]
    fig, ax = plt.subplots()
    im = ax.imshow(acc_matrix, cmap='copper')
    ax.set_xticks(np.arange(len(bit_choices)))
    ax.set_yticks(np.arange(len(bit_choices)))
    ax.set_xticklabels(bit_choices)
    ax.set_yticklabels(bit_choices)
    for i in range(len(bit_choices)):
        for j in range(len(bit_choices)):
            ax.text(j, i, f"{acc_matrix[i, j]:.1f}", ha="center", va="center", color="w")
    ax.set_xlabel("Weight Bit-width")
    ax.set_ylabel("Activation Bit-width")
    ax.set_title(title)
    cbar = plt.colorbar(im)
    cbar.set_label("Accuracy (%)")
    plt.show()


###############################################
# Main Process
###############################################

# Post-Training Quantization Experiment
# Test two quantization methods: tensor-wise, filter-wise
quant_methods = {
    "tensor": tensor_wise_quantize,
    "filter": filter_wise_quantize,  # mode 'channel' indicates using channel-wise quantization
}

all_results = {}
for mode, q_fn in quant_methods.items():
     print(f"\n>>> Post-Training Quantization: {mode} wise <<<")
     results = evaluate_quantized_models(model, dataloader, q_fn, mode, bit_choices=[2, 4, 8])
     all_results[mode] = results
     plot_quantization_results(results, f"Post-Training Quantization ({mode} wise)")


### Question Answering

🔴 **Question:** Have you noticed any differences in the accuracy of the quantized neural networks from 1.1 and 1.2? Can you discuss how the granularity of quantization affects accuracy? (10 points)

🖊 **Answer:**

Filter-wise quantization generally achieves higher accuracy than tensor-wise quantization, especially at lower bit-widths (e.g., 2-bit and 4-bit). This is because filter-wise quantization assigns separate quantization parameters (scale and zero-point) to each output filter, which better captures the unique weight distribution of each filter. In tensor-wise quantization, a single set of parameters is shared across the entire weight tensor, so outlier values in any filter can stretch the quantization range for all filters, leading to larger quantization errors overall.

The finer granularity of filter-wise quantization reduces information loss by adapting to each filter's specific dynamic range. At higher bit-widths (e.g., 8-bit), the difference narrows because there are enough quantization levels to represent the weight distributions accurately regardless of granularity.

# Question 2: Floating Point Representation (10 pts)

In this section, you need to convert $3\times 3$ floating array by showing its sign matrix, exponent matrix, and mantissa matrix. For example, for a vector of $[5.5,2.625,-3.125,2.75]$, their floating point conversion are described as follows:

<p align="center">
  <img src="https://github.com/shawnyin128/25Fall-CSCI-GA-3033-EfficientAI-Materials/raw/master/graph/lab2_fp.png" alt="Magnitude Pruning" width="600"/>
</p>

Therefore the corresponding sign matrix is $[0,0,1,0]$, exponent matrix is $[129,128,128,128]$, and mantissa matrix is $[0.375, 0.3125, 0.5625, 0.375]$, assume the bias is 127.

In [ ]:
import torch
def compute_s_e_m(x: torch.Tensor, bias: int = 127):
    """
    Given a FP32 tensor x, compute the sign(s), exponent (e), and mantissa (m) according to:
      1) a = floor(log2(|x|))
      2) e = a + bias
      3) m = |x| / 2^a - 1
    Note: This ignores the sign bit (s) for now and assumes x != 0.

    :param x: A torch.Tensor of float32 values
    :param bias: The bias used in the exponent, default is 127 for FP32
    :return: (e, m) where both are torch.Tensor
    """
    ##############################################################################
    #TODO
    ##############################################################################
    # step 1: Generate the sign matrix of x and compute the absolute value of x
    # step 2: a = floor(log2(|x|))
    # step 3: e = a + bias
    # step 4: m = |x| / 2^a - 1

    # 1) Compute the absolute value of x
    s = (x < 0).int()
    x_abs = torch.abs(x)

    # We need to avoid taking log2(0). Let's clamp the absolute value to a small positive number.
    # Otherwise, log2(0) is -inf, which we can't use in floor.
    eps = 1e-45  # something extremely small
    x_abs_clamped = torch.clamp(x_abs, min=eps)

    # 2) a = floor(log2(|x|))
    a = torch.floor(torch.log2(x_abs_clamped))

    # 3) e = a + bias
    e = a + bias

    # 4) m = |x| / 2^a - 1
    # we have to handle 2^a carefully, e.g. 2^a = 2^(floor(log2(x_abs))).
    # Use pow(2, a) or 2**a for element-wise exponentiation.
    m = x_abs / torch.pow(2.0, a) - 1.0

    ##############################################################################
    #END OF YOUR CODE                              #
    ##############################################################################
    return s, e, m

In [ ]:
# Print Exponent (e) and Mantissa (m) of a tensor matrix (The outputs are a matrix of Exponent and a matrix of Mantissa)
print(f"\n>>> Exponent (e) and Mantissa (m) of a tensor matrix <<<")

# Creat a 3×3 FP32 Tensor
tensor_FP32 = torch.tensor([
    [-0.125,  1.5,   -2.75  ],
    [0.21875,  -0.325,  1.25],
    [-0.75, -0.3125, 3.25  ]
], dtype=torch.float32)

# Compute Sign(s), Exponent (e), and Mantissa (m)
s_2d, e_2d, m_2d = compute_s_e_m(tensor_FP32)

print("tensor_FP32:")
print(tensor_FP32 )
print("\n sign:")
print(s_2d)
print("\n exponent:")
print(e_2d)
print("\n mantissa:")
print(m_2d)

# Question 3: Symmetric Quantization and Hyperparameter Selection (20 pts)

In this section, you are required to apply tensor-wise symmetric quantization to the CNN weights and analyze accuracy by adjusting the clipping factor.
<p align="center">
  <img src="https://raw.githubusercontent.com/DZY122/Lab-picture/main/quantization.png" alt="Magnitude Pruning" width="400"/>
</p>

Given a floating-point tensor \( $x$ \), symmetric quantization with a clipping factor is performed as follows:

1. **Compute Maximum Absolute Value:**

$$
M = \max_i \left| x_i \right|
$$

2. **Determine Clipping Range Using Clipping Factor (\$\alpha$\):**

$$
R = \alpha \cdot M
$$

3. **Clip the Input Tensor:**

$$
x_{\text{clipped}} = \operatorname{clip}(x, -R, R)
$$

4. **Define the Denominator for Scale Calculation:**

$$
s = 2^{\text{b}} - 2
$$

5. **Compute the Scale Factor:**

$$
\text{scale} = \frac{2R}{s} = \frac{2\alpha M}{2^{\text{b}} - 2}
$$

6. **Quantize to Integer Representation:**

$$
x_{\text{int}} = \operatorname{round}\!\left(\frac{x_{\text{clipped}}}{\text{scale}}\right)
$$

7. **Dequantize to Obtain the Final Quantized Value:**

$$
x_q = x_{\text{int}} \cdot \text{scale}
$$

Following that, you will explore how the clipping factor affects the accuracy of the DNN. We apply an uniform clipping factor across all the weights within the CNN. First, you need to implement the symmetric quantization, which takes the clipping factor as an input.

In [ ]:
def tensor_wise_quantize_with_factor_symmetric(x: torch.Tensor,num_bits: int,clipping_factor: float = 1.0):
    ##############################################################################
    #TODO: implement tensor-wise symmetric quantization with clipping
    ##############################################################################
    # step 1: Using the clipping factor to compute the clipping range, then truncate x using the clipping rate
    M = x.abs().max()
    R = clipping_factor * M
    x_clipped = torch.clamp(x, -R, R)
    # step 2: Calculate the scale s
    s_denom = 2 ** num_bits - 2
    scale = (2 * R) / s_denom
    # step 3: Calculate the fixed-point representation x_{int}
    x_int = torch.round(x_clipped / scale)
    # step 4: Rescale
    x_quant = x_int * scale

    ##############################################################################
    #END OF YOUR CODE                              #
    ##############################################################################

    return x_quant

Then we evaluate the accuracy of DNN under different clipping factors

In [ ]:
def weight_only_quantize_model_with_factor(model: nn.Module,
                w_bit: int,
                factor: float = 1.0) -> nn.Module:
    """

    """
    model_quant = copy.deepcopy(model)
    for name, module in model_quant.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            # Quantize weights
            module.weight.data = tensor_wise_quantize_with_factor_symmetric(module.weight.data, w_bit, factor)

    return model_quant


def evaluate_quantized_models_with_factor(model: nn.Module,dataloader: dict, w_bit=4, factors = [0.8, 0.85, 0.9, 0.95, 1.0]):
    results = {}
    for factor in factors:
      model_q = weight_only_quantize_model_with_factor(model,w_bit=w_bit,factor=factor)
      acc = evaluate(model_q, dataloader['test'])
      results[(w_bit, factor)] = acc
      print(f"[Factor={factor}, W_bit={w_bit}] -> Acc={acc:.2f}%")
    return results


def plot_factor_results(results, w_bit_target, factors):

    acc_list = []
    for f in factors:
        acc = results[(w_bit_target, f)]
        acc_list.append(acc)

    plt.figure()
    plt.plot(factors, acc_list, marker='o', label=f"(w={w_bit_target})")
    plt.xlabel("Clipping Factor")
    plt.ylabel("Accuracy (%)")
    plt.title("Accuracy vs. Clipping Factor (4-bit Quantization)")
    plt.grid(True)
    plt.legend()
    plt.show()


clipping_factors = [0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
all_factor_results = evaluate_quantized_models_with_factor(model, dataloader, w_bit=4, factors=clipping_factors)
plot_factor_results(all_factor_results, w_bit_target=4, factors=clipping_factors)


### Question Answering

🔴 **Question:** Which clipping factor from the set [0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0] do you find to be optimal? Why might employing a clipping factor less than 1.0 in weight quantization result in increased accuracy? What are the potential drawbacks of setting the clipping factor too low or too high? (10 pts)

🖊 **Answer:**

The optimal clipping factor is typically around **0.85-0.9** for 4-bit weight quantization.

Using a clipping factor less than 1.0 can increase accuracy because it reduces the effective quantization range, which means the available quantization levels are more densely packed around the majority of weight values. Since most weights are concentrated near zero (following a Gaussian-like distribution), clipping the outliers allows finer resolution for the bulk of the weights, reducing overall quantization error.

**Drawbacks of too low a clipping factor:** If the factor is too small (e.g., 0.7), too many weights get clipped to the boundary values, causing significant information loss and accuracy degradation. The model essentially loses the contribution of large-magnitude weights that may be important.

**Drawbacks of too high a clipping factor:** If the factor is too close to 1.0, the quantization range is stretched to accommodate outliers, wasting quantization levels on sparsely populated regions of the weight distribution. This reduces precision for the majority of weights near zero, increasing overall quantization error.

# Question 4: Quantization-Aware Training (QAT) (15 pts)

In this section, you are required to implement the Straight Through Estimator (STE) approximation for gradient computation and conduct quantization-aware fine-tuning with 4 bits per tensor for both activations and weights on the ResNet-18 model to restore its accuracy.

The clipping factor for each weight tensor can be different. You need to choose suitable layer-wise clipping factors for the weights to achieve a model accuracy of over 88% with 4-bit weight and 4-bit activation.


## 4.1: Straight-Through Estimator (STE) (10 pts)

As noted in the lecture slides, the quantization function $Q(.)$ is not differentiable, primarily due to that the rounding function Round(.) is not differentiable, which makes backpropagation infeasible.

The **Straight-Through Estimator (STE)** addresses this issue by approximating the derivative of the quantization function with the identity function. In other words, during backpropagation, we assume:
$$
\frac{\partial Round(x)}{\partial x} \approx 1.
$$

This approximation allows gradients to flow through the quantization function despite its non-differentiability.


First, you need to complete your tensor-wise symmetric quantization function by including STE inside, allowing it to be integrated into the training loop.

**Hint:** You can use the `.detach()` (https://pytorch.org/docs/stable/generated/torch.Tensor.detach.html) function, which removes the gradient, preventing it from being backpropagated through the current variable. For example, let $f(x)$ denote some non-differentiable function, and $g(.)$ denotes some differentiable function, then:
$$f(x) + g(x) - g(x).detach()$$
in the training loop indicate that $f(x)$ is applied during the forward propagation, and $g(x)$ is applied during the backpropagation.

In [ ]:
###############################################
# STE Wrapper Function (for Quantization-Aware Training)
###############################################
def tensor_wise_quantize_with_factor_symmetric_ste(x: torch.Tensor,num_bits: int,clipping_factor: float = 1.0):
    ##############################################################################
    #TODO: Conduct STE to round operation
    ##############################################################################
    # step 1: Using the clipping factor to compute the clipping range, then truncate x using the clipping rate
    M = x.abs().max()
    R = clipping_factor * M
    x_clipped = torch.clamp(x, -R, R)
    # step 2: Calculate the scale s
    s_denom = 2 ** num_bits - 2
    scale = (2 * R) / s_denom
    # step 3: Calculate the fixed-point representation x_{int}, with STE enabled
    x_scaled = x_clipped / scale
    x_int = torch.round(x_scaled).detach() + x_scaled - x_scaled.detach()
    # step 4: Rescale
    x_quant = x_int * scale

    ##############################################################################
    #END OF YOUR CODE                             #
    ##############################################################################

    return x_quant

## 4.2: Training (5 pts)
You should adjust the clipping factors for each layer in the CNN by modifying the layer_dict function below. The selected clipping factors should ensure that the quantized CNN achieves an accuracy of over 88%.

In [ ]:
###########################################################################################
#TODO: Please select layer-wise clipping factors for weights to reach the optimal performance
###########################################################################################
layer_dict = {
    'conv1': 0.85,
    'layer1.0.conv1': 0.90,
    'layer1.0.conv2': 0.90,
    'layer1.1.conv1': 0.90,
    'layer1.1.conv2': 0.90,
    'layer2.0.conv1': 0.85,
    'layer2.0.conv2': 0.85,
    'layer2.0.shortcut.0': 0.85,
    'layer2.1.conv1': 0.85,
    'layer2.1.conv2': 0.85,
    'layer3.0.conv1': 0.85,
    'layer3.0.conv2': 0.85,
    'layer3.0.shortcut.0': 0.85,
    'layer3.1.conv1': 0.85,
    'layer3.1.conv2': 0.85,
    'layer4.0.conv1': 0.80,
    'layer4.0.conv2': 0.80,
    'layer4.0.shortcut.0': 0.80,
    'layer4.1.conv1': 0.80,
    'layer4.1.conv2': 0.80,
    'linear': 0.85

}
##############################################################################
#END OF YOUR CODE                              #
##############################################################################

After that, we will train the quantized CNN using 4-bit weights and 4-bit activations.

In [ ]:
def quantize_model_ste(model: nn.Module, w_bit: int, a_bit: int) -> nn.Module:
    """
    Post-Training Quantization of the model:
    - For Conv2d and Linear layers, first quantize the weights (and biases accordingly).
    - Register a forward hook to quantize the activation outputs.

    For each weight parameter, a custom clipping factor is used based on the layer's name.
    If a layer's weight name is not found in layer_dict, the default clipping factor is used.
    """
    model_quant = copy.deepcopy(model)
    for name, module in model_quant.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            # Use a custom clipping factor if available; otherwise, use the default
            clipping_factor = layer_dict[name]

            # Quantize weights using the specific clipping factor
            module.weight.data = tensor_wise_quantize_with_factor_symmetric_ste(module.weight.data, w_bit, clipping_factor)
            # Register a forward hook to quantize the activation outputs
            def forward_pre_hook(module, input, a_bit=a_bit):
                new_input = tuple(
                    tensor_wise_quantize_with_factor_symmetric_ste(x, a_bit, 1.0) if isinstance(x, torch.Tensor) else x
                    for x in input
                )
                return new_input

            module.register_forward_pre_hook(forward_pre_hook)
    return model_quant


def qat_train(model: nn.Module, train_loader: DataLoader, test_loader: DataLoader,
              w_bit: int, a_bit: int, lr: float = 0.01, epochs: int = 10, device: str = "cuda"):
    """
    Fine-tune the model using quantization-aware training:
    - Use STE-based quantization (w_bit for weights, a_bit for activations).
    - Record the test accuracy at each epoch to observe the recovery process.
    """
    model_qat = quantize_model_ste(model, w_bit, a_bit)
    model_qat = model_qat.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model_qat.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[2, 4], gamma=0.1)
    test_acc_list = []
    for epoch in range(epochs):
        model_qat.train()
        train_loss = 0
        correct = 0
        total = 0
        if epoch > 0:
          for inputs, targets in tqdm(train_loader, desc=f"QAT Training Epoch {epoch}"):
              inputs, targets = inputs.to(device), targets.to(device)
              optimizer.zero_grad()
              outputs = model_qat(inputs)
              loss = criterion(outputs, targets)
              loss.backward()
              optimizer.step()
              train_loss += loss.item()
              _, predicted = outputs.max(1)
              total += targets.size(0)
              correct += predicted.eq(targets).sum().item()

          scheduler.step()
        test_acc = evaluate(model_qat, test_loader, device)
        test_acc_list.append(test_acc)
        if epoch > 0:
          print(f"Epoch {epoch}: Training Loss = {train_loss/len(train_loader):.4f}, Test Accuracy = {test_acc:.2f}%")
        else:
          print(f"The initial quantized precision : Test Accuracy = {test_acc:.2f}%")
    return model_qat, test_acc_list

In [ ]:
# Quantization-Aware Training (QAT) Experiment
# Here, using tensor-wise quantization as an example with 4-bit activations and 4-bit weights
print("\n>>> Quantization-Aware Training (QAT): Using STE, 4-bit activations, 4-bit weights <<<")
qat_model, qat_acc_list = qat_train(model, dataloader['train'], dataloader['test'], w_bit=4, a_bit=4, lr=0.001, epochs=6, device="cuda")

plt.figure()
plt.plot(range(0, len(qat_acc_list)), qat_acc_list, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Test Accuracy (%)")
plt.title("Test Accuracy Variation during QAT (Tensor-wise, 4-bit Activations, 4-bit Weights)")
plt.grid(True)
plt.show()

# Question 5: Transformer-Based Language Model Quantization (25 pts)

In this section, we shift from CNNs to the dominant architecture in modern generative AI: the **Transformer**. We will work with **GPT-2** from paper ["Language Models are Unsupervised Multitask Learners"](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf), a decoder-only language model that exemplifies the challenges of modern LLMs. While incredibly capable, these models contain hundreds of millions of parameters, making them computationally expensive and difficult to deploy.

## Initialization

First, we import the necessary packages.


In [ ]:
import math
import inspect
from dataclasses import dataclass
from tqdm import tqdm
import copy

import torch
import torch.nn as nn
from torch.nn import functional as F

import tiktoken

<p align="center">
  <img src="https://github.com/shawnyin128/25Fall-CSCI-GA-3033-EfficientAI-Materials/raw/master/graph/lab2_transformer_block.png" alt="Magnitude Pruning" width="400"/>
</p>

Above is a simple abstraction of the transformer key architecture.

Next, we define the transformer block inside GPT-2. A transformer block contains the following main components:


*   LayerNorm
*   Multi-Head Self-Attention with Causal Mask
*   MLP (Multi-Layer Perceptron, also called Feedforward Network-FFN)
*   Residual Connection







In [ ]:
# Adapted from https://github.com/karpathy/nanoGPT/blob/master/model.py

class LayerNorm(nn.Module):
    """ LayerNorm but with an optional bias. PyTorch doesn't support simply bias=False """

    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        # flash attention make GPU go brrrrr but support is only in PyTorch >= 2.0
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            print("WARNING: using slow attention. Flash Attention requires PyTorch >= 2.0")
            # causal mask to ensure that attention is only applied to the left in the input sequence
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                        .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        if self.flash:
            # efficient attention using Flash Attention CUDA kernels
            y = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True)
        else:
            # manual implementation of attention
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

Then, we define the GPT model with the building blocks of transformer and some other components (e.g. word embedding layer, positional embedding layers, output head for language , etc)

In [ ]:
# Adapted from https://github.com/karpathy/nanoGPT/blob/master/model.py

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304 # GPT-2 vocab_size of 50257, padded up to nearest multiple of 64 for efficiency
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.0
    bias: bool = True # True: bias in Linears and LayerNorms, like GPT-2. False: a bit better and faster

class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # with weight tying when using torch.compile() some warnings get generated:
        # "UserWarning: functional_call was passed multiple values for tied weights.
        # This behavior is deprecated and will be an error in future versions"
        # not 100% sure what this is, so far seems to be harmless. TODO investigate
        self.transformer.wte.weight = self.lm_head.weight # https://paperswithcode.com/method/weight-tying

        # init all weights
        self.apply(self._init_weights)
        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # report number of parameters
        print("number of parameters: %.2fM" % (self.get_num_params()/1e6,))

    def get_num_params(self, non_embedding=True):
        """
        Return the number of parameters in the model.
        For non-embedding count (default), the position embeddings get subtracted.
        The token embeddings would too, except due to the parameter sharing these
        params are actually used as weights in the final layer, so we include them.
        """
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, perp=False):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device) # shape (t)

        # forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (t, n_embd)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            # if we are given some desired targets also calculate the loss
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        elif perp:
            # return the logits for all positions for perplexity computation
            logits = self.lm_head(x)
            loss = None
        else:
            # inference-time mini-optimization: only forward the lm_head on the very last position
            logits = self.lm_head(x[:, [-1], :]) # note: using list [-1] to preserve the time dim
            loss = None

        return logits, loss

    @classmethod
    def from_pretrained(cls, model_type, override_args=None):
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        override_args = override_args or {} # default to empty dict
        # only dropout can be overridden see more notes below
        assert all(k == 'dropout' for k in override_args)
        from transformers import GPT2LMHeadModel
        print("loading weights from pretrained gpt: %s" % model_type)

        # n_layer, n_head and n_embd are determined from model_type
        config_args = {
            'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
            'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
            'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
            'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
        }[model_type]
        print("forcing vocab_size=50257, block_size=1024, bias=True")
        config_args['vocab_size'] = 50257 # always 50257 for GPT model checkpoints
        config_args['block_size'] = 1024 # always 1024 for GPT model checkpoints
        config_args['bias'] = True # always True for GPT model checkpoints
        # we can override the dropout rate, if desired
        if 'dropout' in override_args:
            print(f"overriding dropout rate to {override_args['dropout']}")
            config_args['dropout'] = override_args['dropout']
        # create a from-scratch initialized minGPT model
        config = GPTConfig(**config_args)
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = sd.keys()
        sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # discard this mask / buffer, not a param

        # init a huggingface/transformers model
        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()

        # copy while ensuring all of the parameters are aligned and match in names and shapes
        sd_keys_hf = sd_hf.keys()
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # same, just the mask (buffer)
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        # basically the openai checkpoints use a "Conv1D" module, but we only want to use a vanilla Linear
        # this means that we have to transpose these weights when we import them
        assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                # special treatment for the Conv1D weights we need to transpose
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # vanilla copy over the other parameters
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        return model

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Take a conditioning sequence of indices idx (LongTensor of shape (b,t)) and complete
        the sequence max_new_tokens times, feeding the predictions back into the model each time.
        Most likely you'll want to make sure to be in model.eval() mode of operation for this.
        """
        for _ in range(max_new_tokens):
            # if the sequence context is growing too long we must crop it at block_size
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            # forward the model to get the logits for the index in the sequence
            logits, _ = self(idx_cond)
            # pluck the logits at the final step and scale by desired temperature
            logits = logits[:, -1, :] / temperature
            # optionally crop the logits to only the top k options
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            # apply softmax to convert logits to (normalized) probabilities
            probs = F.softmax(logits, dim=-1)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)
            # append sampled index to the running sequence and continue
            idx = torch.cat((idx, idx_next), dim=1)

        return idx


    @torch.no_grad()
    def compute_perplexity(self, input_ids, stride=512):
        self.eval() # Set the model to evaluation mode

        seq_len = input_ids.size(1)
        nlls = [] # To store negative log-likelihoods for each window

        # Iterate over the sequence with a sliding window
        for begin_loc in range(0, seq_len, stride):
            end_loc = min(begin_loc + self.config.block_size, seq_len)

            # We need at least two tokens to have a target for the first input token
            if end_loc - begin_loc < 2:
                continue

            input_chunk = input_ids[:, begin_loc:end_loc].to(next(self.parameters()).device)

            # The forward pass without targets returns logits
            logits, _ = self(input_chunk, perp=True)

            # For language modeling, the logit at position `i` is used to predict the token at `i+1`.
            # So, we shift the logits and labels to align them for loss calculation.
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = input_chunk[:, 1:].contiguous()

            # Calculate the cross-entropy loss for this chunk
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

            # The negative log-likelihood is the loss multiplied by the number of tokens
            nll = loss * (shift_labels.size(1))
            nlls.append(nll)

        # Calculate the overall perplexity
        # Perplexity = exp(total_negative_log_likelihood / total_tokens)
        total_nll = torch.stack(nlls).sum()
        total_tokens = sum(min(self.config.block_size, seq_len - i) - 1 for i in range(0, seq_len, stride) if min(self.config.block_size, seq_len - i) > 1)

        if total_tokens == 0:
            return float('inf') # Avoid division by zero

        perplexity = torch.exp(total_nll / total_tokens)

        return perplexity.item()

Next, we load the pretrained weights and initialize the tokenizer. GPT-2 comes with a family of parameter sizes ranging from 124M to 1.5B. In this lab we will work with the smallest one for simplicity.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print("Loading model...")
gpt2 = GPT.from_pretrained('gpt2', dict(dropout=0.0)) # {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
gpt2.eval() # Set the model to evaluation mode
gpt2.to(device)
print("Model loaded successfully.")

# Load the tokenizer
print("Loading tokenizer...")
tokenizer = tiktoken.get_encoding("gpt2")
encode = lambda s: tokenizer.encode(s, allowed_special={"<|endoftext|>"})
decode = lambda l: tokenizer.decode(l)
print("Tokenizer loaded successfully.")

We now have a pretrained language model!

GPT-2 runs in an auto-regressive manner, which means it predicts the very next token one by one following the user's prompt.

For a quick try-out, we provide a simple prompt, a token budget (the model will generate the next 50 tokens), and some sampling parameters (e.g. temperature, top-k). Feel free to play with the model yourself!

In [ ]:
prompt = 'New York University is'
max_new_tokens = 50
temperature = 0.8
top_k = 200

# Encode the prompt
start_ids = encode(prompt)
# Create a PyTorch tensor from the token IDs. The unsqueeze(0) adds a batch dimension.
x = (torch.tensor(start_ids, dtype=torch.long, device=device).unsqueeze(0))

print("Generating text...")
with torch.no_grad(): # Inference doesn't require gradient calculations
    y = gpt2.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
    # The output 'y' is a tensor of token IDs.
    generated_text = decode(y[0].tolist())
    print(generated_text)

We use [the LAMBDA benchmark](https://arxiv.org/pdf/1606.06031) for evaluation. For language models, we want to generate sentences that're similar to human languages. In LAMBDA, we prompt the model with a passage of text without the final word, and the task is to correctly predict this missing word.

You can check out the test set on [huggingface](https://huggingface.co/datasets/EleutherAI/lambada_openai).

In [ ]:
from datasets import load_dataset

print("Loading LAMBADA dataset...")
lambda_test = load_dataset('EleutherAI/lambada_openai', "default", split='test')
print("Dataset prepared.")

Now we can define the evaluation helper function and check the accuracy of pretrained weights.

In [ ]:
def evaluate_lambada(model, dataset, device, limit=None):
    """
    Evaluates a GPT model on the LAMBADA dataset's test split.

    Args:
        model (nn.Module): The GPT model to evaluate.
        tokenizer: The tokenizer to use (e.g., from tiktoken).
        device (torch.device): The device to run the evaluation on (e.g., 'cuda').
        limit (int, optional): If set, limits the evaluation to the first `limit` examples.
                               Defaults to None, which evaluates the entire dataset.

    Returns:
        float: The accuracy of the model on the LAMBADA task (0.0 to 1.0).
    """
    # Set the model to evaluation mode
    model.eval()

    if limit:
        dataset = dataset.select(range(limit))

    correct_predictions = 0
    total = len(dataset)

    print(f"Starting evaluation on {total} examples...")
    # Use torch.no_grad() for efficiency as we are not training
    with torch.no_grad():
        # Using tqdm for a progress bar
        for example in tqdm(dataset, desc="Evaluating LAMBADA"):
            # The text is in the 'text' column of the dataset
            text = example['text']
            all_tokens = encode(text)

            # The context is all tokens except the last one
            context_tokens = all_tokens[:-1]
            # The target is the very last token
            target_token = all_tokens[-1]

            # Prepare the input tensor for the model
            context_tensor = torch.tensor(context_tokens, dtype=torch.long).unsqueeze(0).to(device)

            # Get the model's prediction
            logits, _ = model(context_tensor)

            # We only care about the logits for the very last token in the sequence
            last_token_logits = logits[:, -1, :]

            # Find the token with the highest probability (argmax)
            predicted_token = torch.argmax(last_token_logits, dim=-1).item()

            # Check if the prediction was correct
            if predicted_token == target_token:
                correct_predictions += 1

    # Calculate the final accuracy
    accuracy = correct_predictions / total
    return accuracy

In [ ]:
# Evaluate full-precision baseline
lambda_acc = evaluate_lambada(gpt2, lambda_test, device)
print(f"\nAccuracy on LAMBDA test set: {lambda_acc:.4f}")

The full-precision GPT-2-small should have an accuracy around 46%, which reproduces the reported number in the original paper.

## Linear Layer Quantization

The computational core of a Transformer is the matrix multiplication that occurs within its linear (also called fully-connected) layers. These operations, found in both the attention and feed-forward network (or MLP) modules, account for the vast majority of the model's parameters. The fundamental operation is
$$
Y = XW+b
$$

Where $X$ is the input activation tensor, $W$ is the weight matrix, and $b$ is the bias. Quantizing these dense matrix multiplications is an effective way to accelerate LLM inference.

<p align="center">
  <img src="https://github.com/shawnyin128/25Fall-CSCI-GA-3033-EfficientAI-Materials/raw/master/graph/lab2_linearlayer_quant_granularity.png" alt="Quantization Granularity (from SmoothQuant paper)" width="400"/>
</p>

Similar to CNN quantization, linear layer quantization can be performed at different granularities. Specifically, two common strategies are:


1.   **Tensor-wise Quantization**: a single set of quantization parameters (e.g. scale) is applied to the entire weights or activation tensor.
2.   **Vector-wise Quantization**: separate quantization parameters are applied for each token in activation tensor (**Token-wise**) and each channel for weights tensor (**Channel-wise**), as illustrated in the figure above.

In the rest of this section, your will:
* Implement the quantization functions for **Tensor-wise** and Vector-wise (**Token-wise** and **Channel-wise**) granularities respectively. (15 pts)

* Apply quantization methods to weights and activations of the linear layers in transformers, including those in attention projections and feedforward networks.

* Evaluate the accuracy of different quantization strategies. (10 pts)

For computation details of `nn.Linear` module, you can refer to [PyTorch Doc](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html).

You will use the same asymmetric, fixed-point post-training quantization methods as introduced in Question 1.

*For simplicity, in 5.1, you can assume the quantization clipping range matches the maximum and minimum values exactly, no clipping is necessary.*

## 5.1 Tensor-wise Asymmetric Quantization (5 pts)

In [ ]:
###############################################
# Quantization Function Implementations
###############################################

def lm_tensor_wise_quantize(x: torch.Tensor, num_bits: int) -> torch.Tensor:
    ##############################################################################
    #TODO: implement tensor-wise quantization for linear layer
    ##############################################################################
    # step 1: Assume the quantization clipping range matches the maximum and minimum values exactly, no clipping is necessary.
    x_min = x.min()
    x_max = x.max()
    # step 2: Calculate the scale s
    q_min = 0
    q_max = 2 ** num_bits - 1
    s = (x_max - x_min) / (q_max - q_min)
    # step 3: Calculate the fixed-point representation x_{int}
    x_int = torch.round((x - x_min) / s)
    x_int = torch.clamp(x_int, q_min, q_max)
    # step 4: Rescale
    x_quant = x_int * s + x_min
    ##############################################################################
    #END OF YOUR CODE                              #
    ##############################################################################
    return x_quant

## 5.2 Vector-wise Asymmetric Quantization (10 pts)

In [ ]:
def lm_channel_wise_quantize(x: torch.Tensor, num_bits: int) -> torch.Tensor:
    ##############################################################################
    #TODO: implement channel-wise quantization for linear layer
    ##############################################################################
    # step 1: Assume the quantization clipping range matches the maximum and minimum values exactly, no clipping is necessary.
    # step 2: Figure out the input shapes and calculate the scale s, the scale will be different for each channel
    # For weight tensor of shape (out_features, in_features), channel-wise means per output channel (dim=0)
    # For a 2D weight: per-row quantization
    if x.dim() == 2:
        x_min = x.min(dim=1, keepdim=True)[0]
        x_max = x.max(dim=1, keepdim=True)[0]
    else:
        # For higher-dim tensors, quantize along the last dim (channel)
        x_min = x.min(dim=-1, keepdim=True)[0]
        x_max = x.max(dim=-1, keepdim=True)[0]
    q_min = 0
    q_max = 2 ** num_bits - 1
    s = (x_max - x_min) / (q_max - q_min)
    # step 3: Calculate the fixed-point representation x_{int}
    x_int = torch.round((x - x_min) / s)
    x_int = torch.clamp(x_int, q_min, q_max)
    # step 4: Rescale
    x_quant = x_int * s + x_min
    ##############################################################################
    #END OF YOUR CODE                              #
    ##############################################################################
    return x_quant


def lm_token_wise_quantize(x: torch.Tensor, num_bits: int) -> torch.Tensor:
    ##############################################################################
    #TODO: implement token-wise quantization for linear layer
    ##############################################################################
    # step 1: Assume the quantization clipping range matches the maximum and minimum values exactly, no clipping is necessary.
    # step 2: Figure out the input shapes and compute the scale s, the scale will be different for each token
    # For activation tensor of shape (batch, seq_len, hidden), token-wise means per-token (quantize along last dim)
    x_min = x.min(dim=-1, keepdim=True)[0]
    x_max = x.max(dim=-1, keepdim=True)[0]
    q_min = 0
    q_max = 2 ** num_bits - 1
    s = (x_max - x_min) / (q_max - q_min)
    # step 3: Calculate the fixed-point representation x_{int}
    x_int = torch.round((x - x_min) / s)
    x_int = torch.clamp(x_int, q_min, q_max)
    # step 4: Rescale
    x_quant = x_int * s + x_min
    ##############################################################################
    #END OF YOUR CODE                              #
    ##############################################################################
    return x_quant

## 5.3: Evaluation (10 pts)

Now that you have finished the quantization functions, let's evaluate the accuracy of quantized GPT-2 under different granularity and bit width.

**Bitwidth:** 4 bits, 8 bits, 16 bits

**Granularity:**
* tensor-wise quantization for both weights and activations
* channel-wise quantzation for weights, token-wise quantization for activations



In [ ]:
from tqdm import tqdm

def quantize_lm(model: nn.Module, quantize_fn, w_bit: int, a_bit: int, mode: str = 'tensor', quant_head = False) -> nn.Module:
    """
    Post-Training Quantization of the model:
    - For Linear layers, first quantize the weights.
    - Register a forward hook to quantize the activation outputs.

    Parameter mode: If 'channel-token', then we use channel-wise for weights and token-wise for activations;
                    if 'tensor', use tensor-wise for both.
    """
    model_quant = copy.deepcopy(model)
    for name, module in model_quant.named_modules():
        if isinstance(module, (nn.Linear)) and name != "lm_head":
            # Quantize weights
            if mode == "tensor":
                module.weight.data = lm_tensor_wise_quantize(module.weight.data, w_bit)

                def forward_pre_hook(module, input, a_bit=a_bit):
                    new_input = tuple(lm_tensor_wise_quantize(x, a_bit) if isinstance(x, torch.Tensor) else x for x in input)
                    return new_input
            else:
                module.weight.data = lm_channel_wise_quantize(module.weight.data, w_bit)

                def forward_pre_hook(module, input, a_bit=a_bit):
                    new_input = tuple(lm_token_wise_quantize(x, a_bit) if isinstance(x, torch.Tensor) else x for x in input)
                    return new_input

            # Register a forward hook to quantize the activation inputs
            module.register_forward_pre_hook(forward_pre_hook)
    return model_quant


def evaluate_quantized_lm(model: nn.Module, quantize_fn, mode: str, bit_choices = [2, 4, 8], quant_head = False) -> dict:
    """
    Iterate over different bit-width combinations for activations and weights, evaluate the quantized model,
    and return the results as a dictionary.
    """
    results = {}
    for a_bit in bit_choices:
        for w_bit in bit_choices:
            model_q = quantize_lm(model, quantize_fn, w_bit, a_bit, mode=mode, quant_head=quant_head)
            acc = evaluate_lambada(model_q, lambda_test, device)
            results[(a_bit, w_bit)] = acc
            print(f"Quantization mode: {mode} wise, Activation {a_bit} bit, Weight {w_bit} bit -> Accuracy: {acc:.4f}")
    return results


def plot_lm_quantization_results(results: dict, title: str):
    """
    Visualize the test accuracy of the 9 bit-width combinations as a heatmap.
    The x-axis represents the weight bit-width, and the y-axis represents the activation bit-width.
    """
    import matplotlib.pyplot as plt
    bit_choices = sorted({k[0] for k in results.keys()})
    acc_matrix = np.zeros((len(bit_choices), len(bit_choices)))
    for i, a_bit in enumerate(bit_choices):
        for j, w_bit in enumerate(bit_choices):
            acc_matrix[i, j] = results[(a_bit, w_bit)]
    fig, ax = plt.subplots()
    im = ax.imshow(acc_matrix, cmap='copper')
    ax.set_xticks(np.arange(len(bit_choices)))
    ax.set_yticks(np.arange(len(bit_choices)))
    ax.set_xticklabels(bit_choices)
    ax.set_yticklabels(bit_choices)
    for i in range(len(bit_choices)):
        for j in range(len(bit_choices)):
            ax.text(j, i, f"{acc_matrix[i, j]:.4f}", ha="center", va="center", color="w")
    ax.set_xlabel("Weight Bit-width")
    ax.set_ylabel("Activation Bit-width")
    ax.set_title(title)
    plt.colorbar(im)
    plt.show()


###############################################
# Main Process
###############################################

# Post-Training Quantization Experiment
# Test two quantization methods: tensor-wise, filter-wise
quant_methods = {
    "tensor": lm_tensor_wise_quantize,
    "channel": lm_channel_wise_quantize,
}

all_results = {}
for mode, q_fn in quant_methods.items():
     print(f"\n>>> Post-Training Quantization: {mode} wise <<<")
     results = evaluate_quantized_lm(gpt2, q_fn, mode, bit_choices=[4, 8, 16])
     all_results[mode] = results
     plot_lm_quantization_results(results, f"Post-Training Quantization ({mode} wise)")

### Question Answering

🔴 **Question:** How do different bit-width and granularity choice affect the accuracy? Among [4bit, 8bit, 16bit], which one do you find to be the sweetspoint for quantizing GPT2? Why? (10 points)

🖊 **Answer:**

**Effect of bit-width:** Higher bit-widths preserve more accuracy. 16-bit quantization has negligible accuracy loss compared to the original FP32 model. 8-bit quantization typically maintains good accuracy with minor degradation. 4-bit quantization causes more significant accuracy loss, especially with tensor-wise granularity.

**Effect of granularity:** Vector-wise quantization (channel-wise for weights, token-wise for activations) consistently outperforms tensor-wise quantization at the same bit-width. This is because transformers have large variation in activation magnitudes across tokens, and weight distributions can vary significantly across output channels. Vector-wise quantization adapts to these per-vector distributions, reducing quantization error.

**Sweet spot:** **8-bit quantization** is the sweet spot for quantizing GPT-2. It achieves near-lossless accuracy (especially with channel-wise/token-wise granularity) while providing a 4x memory reduction compared to FP32. 16-bit provides minimal compression benefit, while 4-bit introduces too much accuracy degradation for most practical applications without additional techniques like QAT or GPTQ. 8-bit strikes the best balance between model quality and efficiency.